In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
import string

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from nltk.stem.porter import PorterStemmer

# ==========================================
# Step 1 – Load the Dataset
# ==========================================
df = pd.read_csv('Tweets.csv')

# remove unnesasary columns
df = df[["airline_sentiment", "text"]]

# ==========================================
# Step 2 – Preprocess Text
# ==========================================
# To download the data required for NLTK in Colab
nltk.download('stopwords')
nltk.download('punkt')
nltk.download('punkt_tab')

from nltk.corpus import stopwords
ps = PorterStemmer()

def clean_text(text):
    text = text.lower()
    text = re.sub(r'http.?://[^\s]+[\s]?', '', text)
    text = nltk.word_tokenize(text)

    y = []
    for i in text:
        if i not in stopwords.words('english') and i not in string.punctuation:
            y.append(i)

    text = y[:]
    y.clear()

    for i in text:
        y.append(ps.stem(i))

    return " ".join(y)

print("Text is being cleaned... (this may take a minute or two)...")
df['text_cleaned'] = df['text'].apply(clean_text)
print("Preprocessing completed successfully!\n")

# ==========================================
# Step 3 – Feature Extraction
# ==========================================
# Creating a TF-IDF Vectorizer with 3000 maximum features (max_features)
tfidf = TfidfVectorizer(max_features=3000)

# Convert the cleaned text into an array (X)
X = tfidf.fit_transform(df['text_cleaned']).toarray()

# Converting the target (airline_sentiment) into an array (Y)
Y = df['airline_sentiment'].values

# ==========================================
# Step 4 – Train Model
# ==========================================
# Splitting the data into 80% Train and 20% Test (test_size=0.2, random_state=2)
X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=2)

# ---- 1. Multinomial Naive Bayes Classifier ----
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)
nb_pred = nb_model.predict(X_test)
nb_accuracy = accuracy_score(y_test, nb_pred)

print("================ RESULTS ================")
print(f"Multinomial Naive Bayes Accuracy: {nb_accuracy:.4f} (or {nb_accuracy*100:.2f}%)")

# ---- 2. Random Forest Classifier ----
rf_model = RandomForestClassifier(random_state=2)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Random Forest Classifier Accuracy: {rf_accuracy:.4f} (or {rf_accuracy*100:.2f}%)")
print("=========================================")

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


Text is being cleaned... (this may take a minute or two)...
Preprocessing completed successfully!

================ RESULTS ================
Multinomial Naive Bayes Accuracy: 0.7213 (or 72.13%)
Random Forest Classifier Accuracy: 0.7544 (or 75.44%)
